In [1]:
# !pip install -q -U transformers accelerate bitsandbytes langchain langchain-community langchain-huggingface PyPDF2 langchain-classic

In [2]:
import os
import json
import pandas as pd
import traceback
import torch
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline

In [4]:
# 1. Define the model
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

In [5]:
# 2. Configure 4-bit quantization to fit the model in Colab's 16GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [6]:
# 3. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [7]:
# 4. Create the Text Generation Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.5,
    return_full_text=False
)

Device set to use cuda:0


In [8]:
# 5. Wrap in LangChain
llm = HuggingFacePipeline(pipeline=pipe)

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_classic.chains import SequentialChain
import PyPDF2

In [10]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [11]:
TEMPLATE = """
<s>[INST]
You are an expert MCQ maker.
Create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and conform to the text.

Text:
{text}

Format your response strictly as this JSON object:
{response_json}

Output ONLY the JSON.
[/INST]
"""

In [12]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
    )

In [13]:
quiz_chain=LLMChain(llm=llm, prompt=quiz_generation_prompt, output_key="quiz", verbose=True)

/tmp/ipython-input-2669661367.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  quiz_chain=LLMChain(llm=llm, prompt=quiz_generation_prompt, output_key="quiz", verbose=True)


In [14]:
# Cell 66
TEMPLATE2 = """
<s>[INST]
You are an expert English grammarian and writer.
Evaluate the complexity of the following quiz questions:
{quiz}

Provide a complexity analysis (max 50 words).
[/INST]
"""

In [15]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE)

In [16]:
review_chain=LLMChain(llm=llm, prompt=quiz_evaluation_prompt, output_key="review", verbose=True)

In [17]:
generate_evaluate_chain=SequentialChain(chains=[quiz_chain, review_chain], input_variables=["text", "number", "subject", "tone", "response_json"],
                                        output_variables=["quiz", "review"], verbose=True,)

In [18]:
file_path=r"/content/01_data.txt"

In [19]:
with open(file_path, 'r', encoding='utf-8') as file:
    TEXT = file.read()

In [20]:
print(TEXT)

What is machine learning?
Machine learning is the subset of artificial intelligence (AI) focused on algorithms that can “learn” the patterns of training data and, subsequently, make accurate inferences about new data. This pattern recognition ability enables machine learning models to make decisions or predictions without explicit, hard-coded instructions.

Machine learning has come to dominate the field of AI: it provides the backbone of most modern AI systems, from forecasting models to autonomous vehicles to large language models (LLMs) and other generative AI tools.

The central premise of machine learning (ML) is that if you optimize a model’s performance on a dataset of tasks that adequately resemble the real-world problems it will be used for—through a process called model training—the model can make accurate predictions on the new data it sees in its ultimate use case.

Training itself is simply a means to an end: generalization, the translation of strong performance on trainin

In [21]:
# Serialize the Python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [22]:
NUMBER=5
SUBJECT="biology"
TONE="simple"

In [23]:
response = generate_evaluate_chain(
    {
        "text": TEXT,
        "number": NUMBER,
        "subject": SUBJECT,
        "tone": TONE,
        "response_json": json.dumps(RESPONSE_JSON)
    }
)

/tmp/ipython-input-3120652783.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = generate_evaluate_chain(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.




> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

<s>[INST] 
You are an expert MCQ maker. 
Create a quiz of 5 multiple choice questions for biology students in simple tone.
Make sure the questions are not repeated and conform to the text.

Text: 
What is machine learning?
Machine learning is the subset of artificial intelligence (AI) focused on algorithms that can “learn” the patterns of training data and, subsequently, make accurate inferences about new data. This pattern recognition ability enables machine learning models to make decisions or predictions without explicit, hard-coded instructions.

Machine learning has come to dominate the field of AI: it provides the backbone of most modern AI systems, from forecasting models to autonomous vehicles to large language models (LLMs) and other generative AI tools.

The central premise of machine learning (ML) is that if you optimize a model’s performance on a dataset of tasks that adeq

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:

<s>[INST] 
You are an expert MCQ maker. 
Create a quiz of 5 multiple choice questions for biology students in simple tone.
Make sure the questions are not repeated and conform to the text.

Text: 
What is machine learning?
Machine learning is the subset of artificial intelligence (AI) focused on algorithms that can “learn” the patterns of training data and, subsequently, make accurate inferences about new data. This pattern recognition ability enables machine learning models to make decisions or predictions without explicit, hard-coded instructions.

Machine learning has come to dominate the field of AI: it provides the backbone of most modern AI systems, from forecasting models to autonomous vehicles to large language models (LLMs) and other generative AI tools.

The central premise of machine learning (ML) is that if you optimize a model’s performance on a dataset of tasks that adequately resemble the rea

In [24]:
response

{'text': 'What is machine learning?\nMachine learning is the subset of artificial intelligence (AI) focused on algorithms that can “learn” the patterns of training data and, subsequently, make accurate inferences about new data. This pattern recognition ability enables machine learning models to make decisions or predictions without explicit, hard-coded instructions.\n\nMachine learning has come to dominate the field of AI: it provides the backbone of most modern AI systems, from forecasting models to autonomous vehicles to large language models (LLMs) and other generative AI tools.\n\nThe central premise of machine learning (ML) is that if you optimize a model’s performance on a dataset of tasks that adequately resemble the real-world problems it will be used for—through a process called model training—the model can make accurate predictions on the new data it sees in its ultimate use case.\n\nTraining itself is simply a means to an end: generalization, the translation of strong perfo

In [25]:
quiz=response.get("quiz")

In [26]:
# Cell 45: Robust JSON Parsing
import json

# Get the raw output
raw_quiz_output = response.get("quiz", "")

# Find the start and end of the valid JSON object
start_index = raw_quiz_output.find('{')
end_index = raw_quiz_output.rfind('}') + 1

if start_index != -1 and end_index != -1:
    # Extract only the JSON part
    cleaned_quiz_str = raw_quiz_output[start_index:end_index]

    try:
        # Parse the cleaned string
        quiz = json.loads(cleaned_quiz_str)
        print("✅ JSON parsed successfully!")
    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Failed: {e}")
        print("Raw content:", raw_quiz_output)
else:
    print("❌ No JSON brackets found in the output.")
    print("Raw content:", raw_quiz_output)

✅ JSON parsed successfully!


In [27]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [28]:
quiz_table_data

[{'MCQ': 'What is the subset of AI that is focused on algorithms that can learn patterns from training data and make accurate inferences about new data?',
  'Choices': 'a: Deep learning | b: Artificial Intelligence (AI) | c: Machine learning | d: Data science',
  'Correct': 'c'},
 {'MCQ': 'Which of the following is NOT a real-world application of machine learning?',
  'Choices': 'a: Forecasting models | b: Autonomous vehicles | c: Large language models | d: Farming equipment',
  'Correct': 'd'},
 {'MCQ': "What is the process of optimizing a model's performance on a dataset of tasks to make accurate predictions on new data called?",
  'Choices': 'a: Model deployment | b: Model training | c: Model testing | d: Model validation',
  'Correct': 'b'},
 {'MCQ': 'Which of the following is the central premise of machine learning?',
  'Choices': 'a: Generalization is the translation of strong performance on training data to useful results in real-world scenarios | b: Deep learning is the subset 

In [29]:
quiz=pd.DataFrame(quiz_table_data)

In [30]:
quiz.to_csv("01_machinelearning.csv",index=False)